<a href="https://colab.research.google.com/github/EsatovinVictoria/ML_Semester5_EsatovinEbenhaezerVictoria/blob/main/Pertemuan_6/Praktikum_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Praktikum 6

In [1]:
import pandas as pd
import numpy as np
import time
import faiss

from annoy import AnnoyIndex
import hnswlib

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset PBL
# -------------------------------
df = pd.read_csv('burnout_dataset1.csv')

features = [
    'Age',
    'Work_Hours_Per_Week',
    'Job_Satisfaction',
    'Work_Life_Balance',
    'Sleep_Hours',
    'Anxiety_Score',
    'Depression_Score',
    'Screen_Time_Hours',
    'Social_Media_Hours',
    'Coffee_Cups_Per_Day',
    'Absenteeism_Days',
    'Burnout_Score'
]

X = df[features].fillna(df[features].mean(numeric_only=True)).values

print("Jumlah data :", X.shape[0])
print("Jumlah fitur:", X.shape[1])

Jumlah data : 50000
Jumlah fitur: 12


In [2]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Disarankan untuk FAISS dan HNSW
X_scaled = X_scaled.astype(np.float32)

k = 10

In [3]:
start = time.time()

nn = NearestNeighbors(
    n_neighbors=k,
    algorithm='brute',
    metric='euclidean'
)

nn.fit(X_scaled)

dist_exact, idx_exact = nn.kneighbors(X_scaled[:1000])

time_exact = time.time() - start

print(f"Exact NN done in {time_exact:.3f} s")

Exact NN done in 1.904 s


In [4]:
start = time.time()

f = X_scaled.shape[1]

index_annoy = AnnoyIndex(
    f,
    'euclidean'
)

for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)

index_annoy.build(10)

idx_annoy = [
    index_annoy.get_nns_by_vector(v, k)
    for v in X_scaled[:1000]
]

time_annoy = time.time() - start

print(f"Annoy done in {time_annoy:.3f} s")

Annoy done in 0.340 s


Menjalankan HNSW

In [5]:
!pip install hnswlib

In [6]:
import time
import hnswlib
import numpy as np

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('burnout_dataset1.csv')

features = [
    'Age',
    'Work_Hours_Per_Week',
    'Job_Satisfaction',
    'Work_Life_Balance',
    'Sleep_Hours',
    'Anxiety_Score',
    'Depression_Score',
    'Screen_Time_Hours',
    'Social_Media_Hours',
    'Coffee_Cups_Per_Day',
    'Absenteeism_Days',
    'Burnout_Score'
]

X = df[features].fillna(df[features].mean(numeric_only=True)).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

k = 10

print("Jumlah data :", X_scaled.shape[0])
print("Jumlah fitur:", X_scaled.shape[1])

Jumlah data : 50000
Jumlah fitur: 12


In [8]:
start = time.time()

p_hnsw = hnswlib.Index(
    space='l2',
    dim=X_scaled.shape[1]
)

p_hnsw.init_index(
    max_elements=X_scaled.shape[0],
    ef_construction=200,
    M=16
)

p_hnsw.add_items(X_scaled)

p_hnsw.set_ef(200)

idx_hnsw, dist_hnsw = p_hnsw.knn_query(
    X_scaled[:1000],
    k=k
)

time_hnsw = time.time() - start

print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 0.737 s


Menjalankan FAISS IVF

In [9]:
!pip install faiss-cpu

In [10]:
import faiss
import time

In [11]:
start = time.time()

quantizer = faiss.IndexFlatL2(
    X_scaled.shape[1]
)

index_faiss = faiss.IndexIVFFlat(
    quantizer,
    X_scaled.shape[1],
    100,
    faiss.METRIC_L2
)

index_faiss.train(X_scaled)

index_faiss.add(X_scaled)

index_faiss.nprobe = 10

dist_faiss, idx_faiss = index_faiss.search(
    X_scaled[:1000],
    k
)

time_faiss = time.time() - start

print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 0.167 s


Membandingkan Profil Karyawan Terdekat

| Model | Metode | Waktu Komputasi | Top-5 Neighbor |
|---|---|---:|---|
| Exact NN | Brute Force | 1.904 s | [0, 18821, 26738, 29251, 45366] |
| Annoy | Approximate NN | 0.340 s | [0, 18821, 26738, 29251, 20900] |
| HNSW | Graph-based ANN | 0.737 s | [0, 18821, 26738, 29251, 45366] |
| FAISS IVF | Inverted File ANN | 0.167 s | [0, 18821, 26738, 29251, 45366] |
